# 🔥 EnerGIS Scenario Studio

Interaktive Optimierung von industriellen Energiesystemen mit wissenschaftlichen Visualisierungen.

## 🎯 Quick Start

1. Alle Zellen mit **Run All** ausführen
2. Config-Pfade bei Bedarf anpassen
3. Ergebnisse werden automatisch in `saved_workflows/` gespeichert
4. Hochwertige Plots (PDF + SVG) für Publikationen werden erstellt

---

## 📦 Setup & Imports

In [1]:
# Minimal-Bootstrap: Füge Projekt-Root zu sys.path hinzu
import sys
from pathlib import Path

# Finde Projekt-Root
current = Path.cwd()
for candidate in [current] + list(current.parents):
    if (candidate / 'energis').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

# Auto-Setup mit notebook_helpers
from energis.io.notebook_helpers import setup_notebook_environment

PROJECT_ROOT = setup_notebook_environment()
print("\n✅ Setup abgeschlossen")

✅ Projekt-Root: c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat
✅ Matplotlib konfiguriert (Standard backend)
✅ Pandas konfiguriert
✅ Warnings unterdrückt

✅ Setup abgeschlossen


In [ ]:
# Imports
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

from energis.run import rolling_horizon as rh
from energis.config.merge import load_and_merge
from energis.io.notebook_helpers import (
    save_workflow_run,
    display_kpi_summary
)

print("✅ Imports erfolgreich")
print(f"📊 Pandas Version: {pd.__version__}")

## ⚙️ Konfiguration

In [3]:
# Config-Pfade
cfg_paths = [
    "configs/base.yaml",
    "configs/tech_catalog.yaml",
    "configs/sites/default.site.yaml",
    "configs/systems/baseline.system.yaml",
    "configs/scenarios/pf_then_rh.workflow.scenario.yaml",
]

# Optional: Overrides
overrides = None  # z.B.: {"run": {"solver": "glpk"}}

# Prüfen
print("📋 Konfigurationsdateien:")
all_exist = True
for p in cfg_paths:
    full_path = PROJECT_ROOT / p
    exists = full_path.exists()
    symbol = "✅" if exists else "❌"
    print(f"  {symbol} {p}")
    if not exists:
        all_exist = False

if not all_exist:
    raise FileNotFoundError("❌ Nicht alle Config-Dateien gefunden!")

print("\n✅ Konfiguration OK")

📋 Konfigurationsdateien:
  ✅ configs/base.yaml
  ✅ configs/tech_catalog.yaml
  ✅ configs/sites/default.site.yaml
  ✅ configs/systems/baseline.system.yaml
  ✅ configs/scenarios/pf_then_rh.workflow.scenario.yaml

✅ Konfiguration OK


In [4]:
# Config-Vorschau (spezifisch für Scenario Studio)
cfg_preview = load_and_merge(cfg_paths)

print("🔍 Config-Vorschau:")
print(f"  Solver:        {cfg_preview.get('run', {}).get('solver', 'N/A')}")
print(f"  Zeitschritt:   {cfg_preview.get('run', {}).get('dt_h', 'N/A')} h")
print(f"  CO2-Preis:     {cfg_preview.get('costs', {}).get('co2_price_eur_per_t', 'N/A')} EUR/t")
print(f"  Input-Datei:   {cfg_preview.get('site', {}).get('input_xlsx', 'N/A')}")
print(f"  Jahr:          {cfg_preview.get('site', {}).get('year_target', 'N/A')}")

# Systemkomponenten
sys_cfg = cfg_preview.get('system', {})
n_hp = len([hp for hp in sys_cfg.get('heat_pumps', []) if hp.get('enabled', True)])
n_gen = len([k for k,v in sys_cfg.get('generators', {}).items() if v.get('enabled', False)])
storage = sys_cfg.get('storage', {}).get('enabled', False)

print(f"\n🏭 Systemkomponenten:")
print(f"  Wärmepumpen:   {n_hp}")
print(f"  Generatoren:   {n_gen}")
print(f"  Speicher:      {'Ja' if storage else 'Nein'}")

🔍 Config-Vorschau:
  Solver:        gurobi
  Zeitschritt:   1.0 h
  CO2-Preis:     100.0 EUR/t
  Input-Datei:   Import_Data.xlsx
  Jahr:          2023

🏭 Systemkomponenten:
  Wärmepumpen:   4
  Generatoren:   7
  Speicher:      Ja


## 🚀 Optimierung ausführen

In [5]:
%%time
print("="*70)
print("▶ STARTE OPTIMIERUNG")
print("="*70)
print(f"⏰ Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    workflow = rh.run_workflow(cfg_paths, overrides=overrides)
    
    print("\n" + "="*70)
    print("✅ OPTIMIERUNG ERFOLGREICH ABGESCHLOSSEN")
    print("="*70)
    print(f"\n📊 Workflow: {' → '.join(workflow.plan.steps)}")
    
    optimization_successful = True
    
except Exception as e:
    print("\n" + "="*70)
    print("❌ FEHLER BEI DER OPTIMIERUNG")
    print("="*70)
    print(f"\n🔴 Fehler: {str(e)}\n")
    
    import traceback
    print("📋 Vollständiger Traceback:")
    traceback.print_exc()
    
    workflow = None
    optimization_successful = False
    
    print("\n💡 Troubleshooting:")
    print("  1. Prüfe ob Import_Data.xlsx existiert")
    print("  2. Prüfe Solver-Installation (gurobi/glpk)")
    print("  3. Prüfe ob alle Dependencies installiert sind")

▶ STARTE OPTIMIERUNG
⏰ Start: 2025-12-01 05:31:17

[LOAD] Import_Data.xlsx → 8760 Schritte von 2023-01-01 00:00:00 bis 2023-12-31 23:00:00
[BUILD] Using simple storage (single-zone model)
[BUILD] #el_in=5, #el_out=3, #ht_out=12, #ht_in=1
[BUILD] Using simple storage (single-zone model)
[BUILD] #el_in=5, #el_out=3, #ht_out=12, #ht_in=1
[BUILD] Using simple storage (single-zone model)
[BUILD] #el_in=5, #el_out=3, #ht_out=12, #ht_in=1
[BUILD] Using simple storage (single-zone model)
[BUILD] #el_in=5, #el_out=3, #ht_out=12, #ht_in=1
[BUILD] Using simple storage (single-zone model)
[BUILD] #el_in=5, #el_out=3, #ht_out=12, #ht_in=1
[BUILD] Using simple storage (single-zone model)
[BUILD] #el_in=5, #el_out=3, #ht_out=12, #ht_in=1
[BUILD] Using simple storage (single-zone model)
[BUILD] #el_in=5, #el_out=3, #ht_out=12, #ht_in=1
[BUILD] Using simple storage (single-zone model)
[BUILD] #el_in=5, #el_out=3, #ht_out=12, #ht_in=1
[BUILD] Using simple storage (single-zone model)
[BUILD] #el_in=5, #e

## 💾 Workflow speichern & exportieren

In [6]:
if optimization_successful and workflow:
    # Workflow-Name und Beschreibung
    WORKFLOW_NAME = "Scenario Studio Run"
    WORKFLOW_DESCRIPTION = "Interaktive Szenario-Analyse mit wissenschaftlichen Plots"
    
    # Workflow speichern (inkl. CSV, PDF, SVG Exports)
    workflow_dir = save_workflow_run(
        workflow,
        name=WORKFLOW_NAME,
        description=WORKFLOW_DESCRIPTION,
        config_paths=cfg_paths
    )
    
else:
    print("⚠️  Workflow-Speicherung übersprungen (Optimierung fehlgeschlagen)")

📦 Exportiere Ergebnisse (CSV, PDF, SVG)...
📁 Speicherverzeichnis: exports\20251201_054323_PF_THEN_RH-PF-followed-by-rolling-horizon
💾 Speichere Workflow-Objekt...
✅ Workflow gespeichert: workflow.pkl
📝 Erstelle Metadaten...
✅ Metadaten gespeichert: metadata.json
🔄 Verschiebe nach saved_workflows/...
✅ Verschoben nach: saved_workflows\20251201_054323_PF_THEN_RH-PF-followed-by-rolling-horizon

✅ WORKFLOW ERFOLGREICH GESPEICHERT
📂 Verzeichnis: saved_workflows\20251201_054323_PF_THEN_RH-PF-followed-by-rolling-horizon
📊 Dateien:
   • workflow.pkl    - Workflow-Objekt
   • metadata.json   - Metadaten
   • *.csv           - Zeitreihen
   • *.pdf, *.svg    - Plots
   • design.json     - Anlagen-Design


## 📈 Wissenschaftliche Visualisierungen

Erstelle hochwertige Plots in PDF + SVG Format für Publikationen.

In [7]:
if optimization_successful and workflow:
    from energis.io.publication_plotter import export_publication_plots
    
    # Plot-Typen für wissenschaftliche Publikationen
    plot_types = [
        "input_data",           # Wärmebedarf, Strompreis, WRG Temp, WRG Q
        "results_combined",     # Bedarf + Erzeugung + Strompreis
        "heat_balance",         # Wärme-Bilanz
        "electric_balance",     # Elektrische Bilanz
        "storage",              # Speicher-Operation
    ]
    
    print("📊 Erstelle wissenschaftliche Plots (PDF + SVG)...\n")
    
    # Bestimme primäres Ergebnis
    primary_result = workflow.pf_result if workflow.pf_result else workflow.rh_result
    
    if primary_result:
        generated = export_publication_plots(
            outdir=str(workflow_dir),
            table=primary_result.table,
            series=primary_result.series,
            summary_sections=primary_result.summary if hasattr(primary_result, 'summary') else {},
            dpi=300,
            formats=("pdf", "svg"),  # Beide Vektorformate
            plot_types=plot_types
        )
        
        print(f"✅ {len(generated)} Plot-Typen erstellt:")
        for plot_type, files in generated.items():
            print(f"  • {plot_type}: {len(files)} Dateien")
            for f in files:
                print(f"    - {Path(f).name}")
        
        print(f"\n💡 Plots gespeichert in: {workflow_dir}")
        print("\n📚 Verwendung in Publikationen:")
        print("   • PDF: Direkt in LaTeX/Word/PowerPoint einfügen")
        print("   • SVG: Nachbearbeitung in Inkscape/Illustrator möglich")
        print("\n   LaTeX-Beispiel:")
        print("   \\includegraphics[width=\\textwidth]{heat_balance.pdf}")
    else:
        print("⚠️  Keine Ergebnisse für Plot-Erstellung verfügbar")
else:
    print("⚠️  Plot-Erstellung übersprungen (Optimierung fehlgeschlagen)")

📊 Erstelle wissenschaftliche Plots (PDF + SVG)...

✅ 4 Plot-Typen erstellt:
  • results_combined: 2 Dateien
    - results_combined.pdf
    - results_combined.svg
  • heat_balance: 2 Dateien
    - heat_balance_publication.pdf
    - heat_balance_publication.svg
  • electric_balance: 2 Dateien
    - electric_balance_publication.pdf
    - electric_balance_publication.svg
  • storage: 2 Dateien
    - storage_operation_publication.pdf
    - storage_operation_publication.svg

💡 Plots gespeichert in: saved_workflows\20251201_054323_PF_THEN_RH-PF-followed-by-rolling-horizon

📚 Verwendung in Publikationen:
   • PDF: Direkt in LaTeX/Word/PowerPoint einfügen
   • SVG: Nachbearbeitung in Inkscape/Illustrator möglich

   LaTeX-Beispiel:
   \includegraphics[width=\textwidth]{heat_balance.pdf}


## 📊 Key Performance Indicators

Detaillierte KPI-Analyse mit Kostenaufschlüsselung.

In [8]:
if optimization_successful and workflow:
    # Nutze gemeinsame KPI-Funktion
    display_kpi_summary(workflow)
else:
    print("⚠️  Keine KPIs verfügbar")


📊 KEY PERFORMANCE INDICATORS

💰 Wirtschaftlichkeit:
   Gesamtkosten:       12,158,185 EUR

   💶 Detaillierte Aufschlüsselung:
      Brennstoffkosten:           16,491,493 EUR

⚡ Elektrische Energie:
   Netzbezug:                   0 MWh
   Einspeisung:            52,953 MWh

🏭 Komponenten-Auslastung:

   Wärmepumpen:
      HP1     : Ø   3.25 MW | Max   8.95 MW |  3535 h aktiv
      HP2     : Ø   2.00 MW | Max   5.95 MW |  3316 h aktiv
      HP3     : Ø   3.82 MW | Max  10.60 MW |  3566 h aktiv
      HP4     : Ø   1.28 MW | Max   4.45 MW |  2792 h aktiv

   Speicher:
      Max SOC:             0.00 MWh
      Ø SOC:               0.00 MWh



## 🔍 Datenexploration (Optional)

Für detaillierte Datenanalyse und Korrelationen.

In [9]:
if optimization_successful and workflow:
    # Erstelle DataFrame für Exploration
    primary_result = workflow.rh_result or workflow.mpc_result or workflow.pf_result
    
    if primary_result:
        ts = pd.DataFrame({
            'timestamp': primary_result.table.index,
            **{col: primary_result.table.data[col] for col in primary_result.table.columns},
            **primary_result.series,
        })
        ts.set_index('timestamp', inplace=True)
        
        print("📋 Zeitreihen-Daten (erste 10 Zeilen):\n")
        display(ts.head(10))
        
        print("\n📊 Statistische Zusammenfassung:\n")
        display(ts.describe())
        
        # DataFrame für weitere Analysen verfügbar machen
        print("\n💡 Tipp: DataFrame 'ts' ist jetzt für weitere Analysen verfügbar")
    else:
        print("⚠️  Keine Daten für Exploration verfügbar")
else:
    print("⚠️  Datenexploration übersprungen")

📋 Zeitreihen-Daten (erste 10 Zeilen):



,strompreis_EUR_MWh,waermebedarf_MWth,grid_co2_kg_MWh,WRG1_Q_cap,WRG1_T_K,WRG2_Q_cap,WRG2_T_K,WRG3_Q_cap,WRG3_T_K,WRG4_Q_cap,WRG4_T_K,P_buy_MW,P_sell_MW,Q_dump_MWth,HP1_Q_th_MW,HP1_Pel_MW,HP1_on,HP1_Q_wrg_MW,HP1_Q_def_MW,HP1_COP,HP2_Q_th_MW,HP2_Pel_MW,HP2_on,HP2_Q_wrg_MW,HP2_Q_def_MW,HP2_COP,HP3_Q_th_MW,HP3_Pel_MW,HP3_on,HP3_Q_wrg_MW,HP3_Q_def_MW,HP3_COP,HP4_Q_th_MW,HP4_Pel_MW,HP4_on,HP4_Q_wrg_MW,HP4_Q_def_MW,HP4_COP,TES_SOC_MWh,TES_charge_MW,TES_discharge_MW,HKW_Q_th_MW,HKW_fuel_MW,HKW_Pel_MW,GTOST_Q_th_MW,GTOST_fuel_MW,GTOST_Pel_MW,BMHKW_Q_th_MW,BMHKW_fuel_MW,BMHKW_Pel_MW,HWS_Q_th_MW,HWS_fuel_MW,HWW_Q_th_MW,HWW_fuel_MW,AVA_Q_th_MW,AVA_fuel_MW,P2H_Q_th_MW,P2H_Pel_MW
timestamp,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2023-01-01 00:00:00,-5.17,72.30,192.28,0.00,310.15,0.00,316.15,0.00,311.65,0.09,285.03,0.00,0.00,0.00,8.32,2.77,1.00,0.00,8.32,3.00,5.95,1.98,1.00,0.00,5.95,3.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,13.03,26.87,4.76,0.00,0.00,0.00,0.00,45.00,45.00,0.00,0.00
2023-01-01 01:00:00,-1.07,74.60,192.33,0.00,310.25,0.00,316.35,0.00,311.75,0.11,284.95,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,4.87,1.62,1.00,0.00,4.87,3.00,10.60,3.53,1.00,0.00,10.60,3.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,14.13,29.13,5.16,0.00,0.00,0.00,0.00,45.00,45.00,0.00,0.00
2023-01-01 02:00:00,-1.47,75.20,193.13,0.00,310.15,0.00,316.15,0.00,311.55,0.12,285.01,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,5.95,1.98,1.00,0.00,5.95,3.00,5.38,1.79,1.00,0.00,5.38,3.00,4.45,1.48,1.00,0.00,4.45,3.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,14.42,29.72,5.26,0.00,0.00,0.00,0.00,45.00,45.00,0.00,0.00
2023-01-01 03:00:00,-5.08,76.60,199.93,0.00,310.05,0.00,316.15,0.00,311.55,0.11,284.95,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,5.95,1.98,1.00,0.00,5.95,3.00,6.14,2.05,1.00,0.00,6.14,3.00,4.45,1.48,1.00,0.00,4.45,3.00,0.00,0.00,0.00,0.00,0.00,0.00,0.05,0.11,0.04,15.00,30.93,5.47,0.00,0.00,0.00,0.00,45.00,45.00,0.00,0.00
2023-01-01 04:00:00,-4.49,83.10,196.49,0.00,310.15,0.00,316.25,0.00,311.65,0.11,284.95,0.00,0.00,0.00,8.95,2.98,1.00,0.00,8.95,3.00,0.00,0.00,0.00,0.00,0.00,0.00,7.69,2.56,1.00,0.00,7.69,3.00,4.45,1.48,1.00,0.00,4.45,3.00,0.00,0.00,0.00,0.00,0.00,0.00,2.01,4.32,1.55,15.00,30.93,5.47,0.00,0.00,0.00,0.00,45.00,45.00,0.00,0.00
2023-01-01 05:00:00,-5.40,90.60,196.31,0.00,310.05,0.00,316.25,0.00,311.55,0.12,285.01,0.00,0.00,0.00,8.44,2.81,1.00,0.00,8.44,3.00,5.95,1.98,1.00,0.00,5.95,3.00,10.60,3.53,1.00,0.00,10.60,3.00,1.34,0.45,1.00,0.00,1.34,3.00,0.00,0.00,0.00,0.00,0.00,0.00,4.27,9.17,3.30,15.00,30.93,5.47,0.00,0.00,0.00,0.00,45.00,45.00,0.00,0.00
2023-01-01 06:00:00,-5.02,96.40,195.05,0.00,310.05,0.00,316.15,0.00,311.55,0.11,284.95,0.00,0.00,0.00,8.95,2.98,1.00,0.00,8.95,3.00,5.95,1.98,1.00,0.00,5.95,3.00,10.60,3.53,1.00,0.00,10.60,3.00,4.45,1.48,1.00,0.00,4.45,3.00,0.00,0.00,0.00,0.88,1.18,0.21,5.57,11.95,4.30,15.00,30.93,5.47,0.00,0.00,0.00,0.00,45.00,45.00,0.00,0.00
2023-01-01 07:00:00,-1.30,100.70,204.27,0.00,310.15,0.00,316.35,0.00,311.75,0.10,284.95,0.00,0.00,0.00,8.95,2.98,1.00,0.00,8.95,3.00,5.95,1.98,1.00,0.00,5.95,3.00,10.60,3.53,1.00,0.00,10.60,3.00,4.45,1.48,1.00,0.00,4.45,3.00,0.00,0.00,0.00,7.10,9.55,1.69,3.65,7.83,2.82,15.00,30.93,5.47,0.00,0.00,0.00,0.00,45.00,45.00,0.00,0.00
2023-01-01 08:00:00,-1.44,105.10,195.38,0.00,310.05,0.00,316.05,0.00,311.55,0.10,284.95,0.00,0.00,0.00,8.95,2.98,1.00,0.00,8.95,3.00,5.95,1.98,1.00,0.00,5.95,3.00,10.60,3.53,1.00,0.00,10.60,3.00,4.45,1.48,1.00,0.00,4.45,3.00,0.00,0.00,0.00,13.46,18.11,3.21,1.69,3.62,1.30,15.00,30.93,5.47,0.00,0.00,0.00,0.00,45.00,45.00,0.00,0.00



📊 Statistische Zusammenfassung:



,strompreis_EUR_MWh,waermebedarf_MWth,grid_co2_kg_MWh,WRG1_Q_cap,WRG1_T_K,WRG2_Q_cap,WRG2_T_K,WRG3_Q_cap,WRG3_T_K,WRG4_Q_cap,WRG4_T_K,P_buy_MW,P_sell_MW,Q_dump_MWth,HP1_Q_th_MW,HP1_Pel_MW,HP1_on,HP1_Q_wrg_MW,HP1_Q_def_MW,HP1_COP,HP2_Q_th_MW,HP2_Pel_MW,HP2_on,HP2_Q_wrg_MW,HP2_Q_def_MW,HP2_COP,HP3_Q_th_MW,HP3_Pel_MW,HP3_on,HP3_Q_wrg_MW,HP3_Q_def_MW,HP3_COP,HP4_Q_th_MW,HP4_Pel_MW,HP4_on,HP4_Q_wrg_MW,HP4_Q_def_MW,HP4_COP,TES_SOC_MWh,TES_charge_MW,TES_discharge_MW,HKW_Q_th_MW,HKW_fuel_MW,HKW_Pel_MW,GTOST_Q_th_MW,GTOST_fuel_MW,GTOST_Pel_MW,BMHKW_Q_th_MW,BMHKW_fuel_MW,BMHKW_Pel_MW,HWS_Q_th_MW,HWS_fuel_MW,HWW_Q_th_MW,HWW_fuel_MW,AVA_Q_th_MW,AVA_fuel_MW,P2H_Q_th_MW,P2H_Pel_MW
count,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00,8760.00
mean,95.19,65.21,254.51,0.80,322.06,0.72,321.94,0.81,322.51,4.17,292.09,0.00,6.04,0.07,3.25,1.07,0.40,0.20,3.05,1.23,2.00,0.66,0.38,0.20,1.81,1.17,3.82,1.26,0.41,0.22,3.60,1.24,1.28,0.43,0.32,0.00,1.28,0.96,0.00,0.00,0.00,4.41,5.93,1.05,5.16,11.08,3.99,12.13,25.01,4.43,1.19,1.27,0.06,0.07,31.97,31.97,0.00,0.00
std,47.58,42.94,126.14,0.98,12.09,0.82,10.02,0.96,10.92,6.51,4.89,0.00,9.74,1.75,4.13,1.37,0.49,0.54,3.98,1.50,2.72,0.90,0.49,0.52,2.56,1.50,4.87,1.61,0.49,0.60,4.71,1.50,1.96,0.65,0.47,0.00,1.96,1.40,0.00,0.00,0.00,11.91,16.03,2.84,12.28,26.36,9.49,5.71,11.77,2.08,6.03,6.45,1.00,1.08,15.76,15.76,0.07,0.07
min,-500.00,11.30,39.64,0.00,273.15,0.00,273.15,0.00,273.15,0.00,273.15,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,75.89,24.10,155.70,0.00,307.75,0.00,311.25,0.00,310.55,0.53,289.11,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,15.00,30.93,5.47,0.00,0.00,0.00,0.00,19.10,19.10,0.00,0.00
50%,98.03,54.30,225.21,0.00,326.65,0.07,327.05,0.05,327.85,1.26,292.21,0.00,4.87,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,15.00,30.93,5.47,0.00,0.00,0.00,0.00,43.80,43.80,0.00,0.00
75%,122.12,98.80,331.63,1.68,329.95,1.59,329.65,1.81,330.55,4.56,295.39,0.00,5.47,0.00,8.95,2.92,1.00,0.00,7.71,3.00,5.95,1.91,1.00,0.00,4.54,3.00,10.60,3.44,1.00,0.00,9.13,3.00,4.45,1.48,1.00,0.00,4.45,3.00,0.00,0.00,0.00,0.00,0.00,0.00,1.23,2.64,0.95,15.00,30.93,5.47,0.00,0.00,0.00,0.00,45.00,45.00,0.00,0.00
max,524.27,191.10,645.37,3.38,344.75,2.46,337.75,3.17,339.45,50.97,304.40,0.00,55.25,105.90,8.95,2.98,1.00,3.18,8.95,4.31,5.95,1.98,1.00,2.19,5.95,4.11,10.60,3.53,1.00,2.73,10.60,3.93,4.45,1.48,1.00,0.00,4.45,3.00,0.00,0.00,0.00,75.00,100.94,17.87,41.30,88.63,31.91,15.00,30.93,5.47,45.00,48.08,30.51,33.02,45.00,45.00,4.70,4.75



💡 Tipp: DataFrame 'ts' ist jetzt für weitere Analysen verfügbar


In [10]:
# Korrelationsmatrix (optional)
if optimization_successful and workflow and 'ts' in locals():
    import matplotlib.pyplot as plt
    
    numeric_cols = ts.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 1:
        fig, ax = plt.subplots(figsize=(12, 10))
        corr = ts[numeric_cols].corr()
        im = ax.imshow(corr, cmap='RdYlBu_r', aspect='auto', vmin=-1, vmax=1)
        
        ax.set_xticks(range(len(corr.columns)))
        ax.set_yticks(range(len(corr.columns)))
        ax.set_xticklabels(corr.columns, rotation=90, ha='right')
        ax.set_yticklabels(corr.columns)
        
        plt.colorbar(im, ax=ax)
        ax.set_title('🔗 Korrelationsmatrix', fontsize=16, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()
else:
    print("⏭️  Korrelationsmatrix übersprungen")

## 🎛️ Dashboard zur Visualisierung

Um die gespeicherten Simulationsergebnisse zu visualisieren, verwende das **Standalone Dashboard**.

### Dashboard starten:

**Option 1: Python-Skript (empfohlen)**
```bash
python start_dashboard.py
```

**Option 2: Workflow Browser Notebook**
```bash
panel serve notebooks/workflow_browser.ipynb --show
```

Das Dashboard läuft unabhängig von der Simulation und lädt automatisch alle gespeicherten Workflows aus `saved_workflows/`.

In [ ]:
print("📊 Simulationsergebnisse wurden gespeichert!")
print("\n🎛️ Zum Visualisieren starte das Dashboard:")
print("   python start_dashboard.py")
print("\nOder verwende das Workflow Browser Notebook:")
print("   panel serve notebooks/workflow_browser.ipynb --show")
print("\n💡 Dashboard-Features:")
print("   • Automatisches Scannen aller Workflows in 'saved_workflows/'")
print("   • Dropdown zur Workflow-Auswahl")
print("   • Vollständiges Dashboard mit allen Tabs")
print("   • Interaktive Plots: Zoom, Pan, Hover")
print("   • Vergleichs-Modus für multiple Simulationen")

In [12]:
dashboard

Column(sizing_mode='stretch_width')
    [0] Markdown(str, sizing_mode='stretch_width')
    [1] Tabs(dynamic=True, sizing_mode='stretch_width')
        [0] Column(sizing_mode='stretch_width')
            [0] Markdown(str, sizing_mode='stretch_width')
            [1] GridBox(ncols=3, sizing_mode='stretch_width')
                [0] HTML(str, sizing_mode='stretch_width')
                [1] HTML(str, sizing_mode='stretch_width')
                [2] HTML(str, sizing_mode='stretch_width')
                [3] HTML(str, sizing_mode='stretch_width')
                [4] HTML(str, sizing_mode='stretch_width')
                [5] HTML(str, sizing_mode='stretch_width')
            [2] Divider(sizing_mode='stretch_width')
            [3] Row(sizing_mode='stretch_width')
                [0] Column(width=400)
                    [0] Markdown(str, sizing_mode='stretch_width')
                    [1] Markdown(str, sizing_mode='stretch_width')
                [1] Column(sizing_mode='stretch_width')
                    [0] Markdown(str, sizing_mode='stretch_width')
                    [1] Plotly(Figure, sizing_mode='stretch_width')
        [1] Column(sizing_mode='stretch_width')
            [0] Card(sizing_mode='stretch_width', title='⚙️ Steuerung')
                [0] MultiChoice(name='🔥 Thermische K..., options=['HP1_Q_th_MW', ...], sizing_mode='stretch_width', value=['HP1_Q_th_MW', ...])
                [1] IntRangeSlider(end=8760, name='📅 Zeitbereich (Stunden)', sizing_mode='stretch_width', step=24, value=(0, 168), value_end=168)
                [2] Select(name='Plot-Typ', options=['Stacked Area', ...], sizing_mode='stretch_width', value='Stacked Area')
            [1] Column(sizing_mode='stretch_width')
                [0] Markdown(str, sizing_mode='stretch_width')
                [1] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
                [2] Divider(sizing_mode='stretch_width')
                [3] Markdown(str, sizing_mode='stretch_width')
                [4] Plotly(Figure, sizing_mode='stretch_width')
                [5] Divider(sizing_mode='stretch_width')
                [6] Markdown(str, sizing_mode='stretch_width')
                [7] Plotly(Figure, sizing_mode='stretch_width')
        [2] Column(sizing_mode='stretch_width')
            [0] Row(sizing_mode='stretch_width')
                [0] Column(sizing_mode='stretch_width')
                    [0] Markdown(str, sizing_mode='stretch_width')
                    [1] Plotly(Figure, sizing_mode='stretch_width')
                [1] Column(width=300)
                    [0] Markdown(str, sizing_mode='stretch_width')
                    [1] Markdown(str, sizing_mode='stretch_width')
            [1] Divider(sizing_mode='stretch_width')
            [2] Markdown(str, sizing_mode='stretch_width')
            [3] Tabulator(formatters={'Value_EUR': {'type': 'mo...}, page_size=20, show_index=False, sizing_mode='stretch_width', theme='modern', value=              ...)
        [3] Column(sizing_mode='stretch_width')
            [0] Markdown(str, sizing_mode='stretch_width')
            [1] Plotly(Figure, sizing_mode='stretch_width')
            [2] Divider(sizing_mode='stretch_width')
            [3] Markdown(str, sizing_mode='stretch_width')
            [4] Tabulator(formatters={'Kapazität [MW]': {'type'...}, show_index=False, sizing_mode='stretch_width', theme='modern', value=             K...)
            [5] Divider(sizing_mode='stretch_width')
            [6] Markdown(str, sizing_mode='stretch_width')
            [7] JSON(str, depth=2, sizing_mode='stretch_width')
        [4] Column(sizing_mode='stretch_width')
            [0] Markdown(str, sizing_mode='stretch_width')
            [1] Plotly(Figure, sizing_mode='stretch_width')
            [2] Divider(sizing_mode='stretch_width')
            [3] Markdown(str, sizing_mode='stretch_width')
            [4] Tabulator(show_index=False, sizing_mode='stretch_width', theme='modern', value=           Metri

---

## 🎯 Nächste Schritte

### Sensitivitätsanalysen:
- CO2-Preis variieren
- Komponenten aktivieren/deaktivieren  
- Kapazitäten anpassen

### Weitere Analysen:
- Jahresdauerlinie erstellen
- Monats-Aggregation
- COP-Entwicklung analysieren

### Export & Sharing:
- Plots wurden als PDF+SVG für Publikationen exportiert
- Workflow wurde in `saved_workflows/` gespeichert
- Dashboard kann als Webapp gestartet werden: `panel serve scenario_studio.ipynb --show`

---